# 02 — Mode Collapse / 02 Mode Collapse

**Chapter 10 — File 2 of 4 / 第10章 — 第2个文件（共4个）**

---

## Summary / 总结

This script demonstrates **example of training an unstable gan for generating a handwritten digit**.

本脚本演示 **example of training an unstable gan for generating a handwritten digit**。

---
## Step 1 — example of training an unstable gan for generating a handwritten digit

In [ ]:
from os import makedirs
from numpy import expand_dims
from numpy import zeros
from numpy import ones
from numpy.random import randn
from numpy.random import randint
from keras.datasets.mnist import load_data
from keras.optimizers import Adam
from keras.models import Sequential
from keras.layers import Dense
from keras.layers import Reshape
from keras.layers import Flatten
from keras.layers import Conv2D
from keras.layers import Conv2DTranspose
from keras.layers import LeakyReLU
from keras.initializers import RandomNormal
from matplotlib import pyplot

---
## Step 2 — define the standalone discriminator model

In [ ]:
def define_discriminator(in_shape=(28,28,1)):

---
## Step 3 — weight initialization

In [ ]:
init = RandomNormal(stddev=0.02)

---
## Step 4 — define model

In [ ]:
model = Sequential()

---
## Step 5 — downsample to 14x14

In [ ]:
model.add(Conv2D(64, (4,4), strides=(2,2), padding='same', kernel_initializer=init, input_shape=in_shape))
	model.add(LeakyReLU(alpha=0.2))

---
## Step 6 — downsample to 7x7

In [ ]:
model.add(Conv2D(64, (4,4), strides=(2,2), padding='same', kernel_initializer=init))
	model.add(LeakyReLU(alpha=0.2))

---
## Step 7 — classifier

In [ ]:
model.add(Flatten())
	model.add(Dense(1, activation='sigmoid'))

---
## Step 8 — compile model

In [ ]:
opt = Adam(lr=0.0002, beta_1=0.5)
	model.compile(loss='binary_crossentropy', optimizer=opt, metrics=['accuracy'])
	return model

---
## Step 9 — define the standalone generator model

In [ ]:
def define_generator(latent_dim):

---
## Step 10 — weight initialization

In [ ]:
init = RandomNormal(stddev=0.02)

---
## Step 11 — define model

In [ ]:
model = Sequential()

---
## Step 12 — foundation for 7x7 image

In [ ]:
n_nodes = 128 * 7 * 7
	model.add(Dense(n_nodes, kernel_initializer=init, input_dim=latent_dim))
	model.add(LeakyReLU(alpha=0.2))
	model.add(Reshape((7, 7, 128)))

---
## Step 13 — upsample to 14x14

In [ ]:
model.add(Conv2DTranspose(128, (4,4), strides=(2,2), padding='same', kernel_initializer=init))
	model.add(LeakyReLU(alpha=0.2))

---
## Step 14 — upsample to 28x28

In [ ]:
model.add(Conv2DTranspose(128, (4,4), strides=(2,2), padding='same', kernel_initializer=init))
	model.add(LeakyReLU(alpha=0.2))

---
## Step 15 — output 28x28x1

In [ ]:
model.add(Conv2D(1, (7,7), activation='tanh', padding='same', kernel_initializer=init))
	return model

---
## Step 16 — define the combined generator and discriminator model, for updating the generator

In [ ]:
def define_gan(generator, discriminator):

---
## Step 17 — make weights in the discriminator not trainable

In [ ]:
discriminator.trainable = False

---
## Step 18 — connect them

In [ ]:
model = Sequential()

---
## Step 19 — add generator

In [ ]:
model.add(generator)

---
## Step 20 — add the discriminator

In [ ]:
model.add(discriminator)

---
## Step 21 — compile model

In [ ]:
opt = Adam(lr=0.0002, beta_1=0.5)
	model.compile(loss='binary_crossentropy', optimizer=opt)
	return model

---
## Step 22 — load mnist images

In [ ]:
def load_real_samples():

---
## Step 23 — load dataset

In [ ]:
(trainX, trainy), (_, _) = load_data()

---
## Step 24 — expand to 3d, e.g. add channels

In [ ]:
X = expand_dims(trainX, axis=-1)

---
## Step 25 — select all of the examples for a given class

In [ ]:
selected_ix = trainy == 8
	X = X[selected_ix]

---
## Step 26 — convert from ints to floats

In [ ]:
X = X.astype('float32')

---
## Step 27 — scale from [0,255] to [-1,1]

In [ ]:
X = (X - 127.5) / 127.5
	return X

---
## Step 28 — # select real samples

In [ ]:
def generate_real_samples(dataset, n_samples):

---
## Step 29 — choose random instances

In [ ]:
ix = randint(0, dataset.shape[0], n_samples)

---
## Step 30 — select images

In [ ]:
X = dataset[ix]

---
## Step 31 — generate class labels

In [ ]:
y = ones((n_samples, 1))
	return X, y

---
## Step 32 — generate points in latent space as input for the generator

In [ ]:
def generate_latent_points(latent_dim, n_samples):

---
## Step 33 — generate points in the latent space

In [ ]:
x_input = randn(latent_dim * n_samples)

---
## Step 34 — reshape into a batch of inputs for the network

In [ ]:
x_input = x_input.reshape(n_samples, latent_dim)
	return x_input

---
## Step 35 — use the generator to generate n fake examples, with class labels

In [ ]:
def generate_fake_samples(generator, latent_dim, n_samples):

---
## Step 36 — generate points in latent space

In [ ]:
x_input = generate_latent_points(latent_dim, n_samples)

---
## Step 37 — predict outputs

In [ ]:
X = generator.predict(x_input)

---
## Step 38 — create class labels

In [ ]:
y = zeros((n_samples, 1))
	return X, y

---
## Step 39 — generate samples and save as a plot and save the model

In [ ]:
def summarize_performance(step, g_model, latent_dim, n_samples=100):

---
## Step 40 — prepare fake examples

In [ ]:
X, _ = generate_fake_samples(g_model, latent_dim, n_samples)

---
## Step 41 — scale from [-1,1] to [0,1]

In [ ]:
X = (X + 1) / 2.0

---
## Step 42 — plot images

In [ ]:
for i in range(10 * 10):

---
## Step 43 — define subplot

In [ ]:
pyplot.subplot(10, 10, 1 + i)

---
## Step 44 — turn off axis

In [ ]:
pyplot.axis('off')

---
## Step 45 — plot raw pixel data

In [ ]:
pyplot.imshow(X[i, :, :, 0], cmap='gray_r')

---
## Step 46 — save plot to file

In [ ]:
pyplot.savefig('results_collapse/generated_plot_%03d.png' % (step+1))
	pyplot.close()

---
## Step 47 — save the generator model

In [ ]:
g_model.save('results_collapse/model_%03d.h5' % (step+1))

---
## Step 48 — create a line plot of loss for the gan and save to file

In [ ]:
def plot_history(d1_hist, d2_hist, g_hist, a1_hist, a2_hist):

---
## Step 49 — plot loss

In [ ]:
pyplot.subplot(2, 1, 1)
	pyplot.plot(d1_hist, label='d-real')
	pyplot.plot(d2_hist, label='d-fake')
	pyplot.plot(g_hist, label='gen')
	pyplot.legend()

---
## Step 50 — plot discriminator accuracy

In [ ]:
pyplot.subplot(2, 1, 2)
	pyplot.plot(a1_hist, label='acc-real')
	pyplot.plot(a2_hist, label='acc-fake')
	pyplot.legend()

---
## Step 51 — save plot to file

In [ ]:
pyplot.savefig('results_collapse/plot_line_plot_loss.png')
	pyplot.close()

---
## Step 52 — train the generator and discriminator

In [ ]:
def train(g_model, d_model, gan_model, dataset, latent_dim, n_epochs=10, n_batch=128):

---
## Step 53 — calculate the number of batches per epoch

In [ ]:
bat_per_epo = int(dataset.shape[0] / n_batch)

---
## Step 54 — calculate the total iterations based on batch and epoch

In [ ]:
n_steps = bat_per_epo * n_epochs

---
## Step 55 — calculate the number of samples in half a batch

In [ ]:
half_batch = int(n_batch / 2)

---
## Step 56 — prepare lists for storing stats each iteration

In [ ]:
d1_hist, d2_hist, g_hist, a1_hist, a2_hist = list(), list(), list(), list(), list()

---
## Step 57 — manually enumerate epochs

In [ ]:
for i in range(n_steps):

---
## Step 58 — get randomly selected 'real' samples

In [ ]:
X_real, y_real = generate_real_samples(dataset, half_batch)

---
## Step 59 — update discriminator model weights

In [ ]:
d_loss1, d_acc1 = d_model.train_on_batch(X_real, y_real)

---
## Step 60 — generate 'fake' examples

In [ ]:
X_fake, y_fake = generate_fake_samples(g_model, latent_dim, half_batch)

---
## Step 61 — update discriminator model weights

In [ ]:
d_loss2, d_acc2 = d_model.train_on_batch(X_fake, y_fake)

---
## Step 62 — prepare points in latent space as input for the generator

In [ ]:
X_gan = generate_latent_points(latent_dim, n_batch)

---
## Step 63 — create inverted labels for the fake samples

In [ ]:
y_gan = ones((n_batch, 1))

---
## Step 64 — update the generator via the discriminator's error

In [ ]:
g_loss = gan_model.train_on_batch(X_gan, y_gan)

---
## Step 65 — summarize loss on this batch

In [ ]:
print('>%d, d1=%.3f, d2=%.3f g=%.3f, a1=%d, a2=%d' %
			(i+1, d_loss1, d_loss2, g_loss, int(100*d_acc1), int(100*d_acc2)))

---
## Step 66 — record history

In [ ]:
d1_hist.append(d_loss1)
		d2_hist.append(d_loss2)
		g_hist.append(g_loss)
		a1_hist.append(d_acc1)
		a2_hist.append(d_acc2)

---
## Step 67 — evaluate the model performance every 'epoch'

In [ ]:
if (i+1) % bat_per_epo == 0:
			summarize_performance(i, g_model, latent_dim)
	plot_history(d1_hist, d2_hist, g_hist, a1_hist, a2_hist)

---
## Step 68 — make folder for results

In [ ]:
makedirs('results_collapse', exist_ok=True)

---
## Step 69 — size of the latent space

In [ ]:
latent_dim = 1

---
## Step 70 — create the discriminator

In [ ]:
discriminator = define_discriminator()

---
## Step 71 — create the generator

In [ ]:
generator = define_generator(latent_dim)

---
## Step 72 — create the gan

In [ ]:
gan_model = define_gan(generator, discriminator)

---
## Step 73 — load image data

In [ ]:
dataset = load_real_samples()
print(dataset.shape)

---
## Step 74 — train model

In [ ]:
train(generator, discriminator, gan_model, dataset, latent_dim)

---
## Learning Notes / 学习笔记

- **概念**: example of training an unstable gan for generating a handwritten digit 是机器学习中的常用技术。  
  *example of training an unstable gan for generating a handwritten digit is a common technique in machine learning.*

- **ML 应用**: 本示例展示了如何在实践中应用该技术。  
  *This example shows how to apply the technique in practice.*

---
## Complete Code / 完整代码一览

Below is the full code for quick reference. / 以下是完整代码，供快速参考。

In [ ]:
# ===============================
# Mode Collapse / 02 Mode Collapse
# Complete Code / 完整代码
# ===============================

# example of training an unstable gan for generating a handwritten digit
from os import makedirs
from numpy import expand_dims
from numpy import zeros
from numpy import ones
from numpy.random import randn
from numpy.random import randint
from keras.datasets.mnist import load_data
from keras.optimizers import Adam
from keras.models import Sequential
from keras.layers import Dense
from keras.layers import Reshape
from keras.layers import Flatten
from keras.layers import Conv2D
from keras.layers import Conv2DTranspose
from keras.layers import LeakyReLU
from keras.initializers import RandomNormal
from matplotlib import pyplot

# define the standalone discriminator model
def define_discriminator(in_shape=(28,28,1)):
	# weight initialization
	init = RandomNormal(stddev=0.02)
	# define model
	model = Sequential()
	# downsample to 14x14
	model.add(Conv2D(64, (4,4), strides=(2,2), padding='same', kernel_initializer=init, input_shape=in_shape))
	model.add(LeakyReLU(alpha=0.2))
	# downsample to 7x7
	model.add(Conv2D(64, (4,4), strides=(2,2), padding='same', kernel_initializer=init))
	model.add(LeakyReLU(alpha=0.2))
	# classifier
	model.add(Flatten())
	model.add(Dense(1, activation='sigmoid'))
	# compile model
	opt = Adam(lr=0.0002, beta_1=0.5)
	model.compile(loss='binary_crossentropy', optimizer=opt, metrics=['accuracy'])
	return model

# define the standalone generator model
def define_generator(latent_dim):
	# weight initialization
	init = RandomNormal(stddev=0.02)
	# define model
	model = Sequential()
	# foundation for 7x7 image
	n_nodes = 128 * 7 * 7
	model.add(Dense(n_nodes, kernel_initializer=init, input_dim=latent_dim))
	model.add(LeakyReLU(alpha=0.2))
	model.add(Reshape((7, 7, 128)))
	# upsample to 14x14
	model.add(Conv2DTranspose(128, (4,4), strides=(2,2), padding='same', kernel_initializer=init))
	model.add(LeakyReLU(alpha=0.2))
	# upsample to 28x28
	model.add(Conv2DTranspose(128, (4,4), strides=(2,2), padding='same', kernel_initializer=init))
	model.add(LeakyReLU(alpha=0.2))
	# output 28x28x1
	model.add(Conv2D(1, (7,7), activation='tanh', padding='same', kernel_initializer=init))
	return model

# define the combined generator and discriminator model, for updating the generator
def define_gan(generator, discriminator):
	# make weights in the discriminator not trainable
	discriminator.trainable = False
	# connect them
	model = Sequential()
	# add generator
	model.add(generator)
	# add the discriminator
	model.add(discriminator)
	# compile model
	opt = Adam(lr=0.0002, beta_1=0.5)
	model.compile(loss='binary_crossentropy', optimizer=opt)
	return model

# load mnist images
def load_real_samples():
	# load dataset
	(trainX, trainy), (_, _) = load_data()
	# expand to 3d, e.g. add channels
	X = expand_dims(trainX, axis=-1)
	# select all of the examples for a given class
	selected_ix = trainy == 8
	X = X[selected_ix]
	# convert from ints to floats
	X = X.astype('float32')
	# scale from [0,255] to [-1,1]
	X = (X - 127.5) / 127.5
	return X

# # select real samples
def generate_real_samples(dataset, n_samples):
	# choose random instances
	ix = randint(0, dataset.shape[0], n_samples)
	# select images
	X = dataset[ix]
	# generate class labels
	y = ones((n_samples, 1))
	return X, y

# generate points in latent space as input for the generator
def generate_latent_points(latent_dim, n_samples):
	# generate points in the latent space
	x_input = randn(latent_dim * n_samples)
	# reshape into a batch of inputs for the network
	x_input = x_input.reshape(n_samples, latent_dim)
	return x_input

# use the generator to generate n fake examples, with class labels
def generate_fake_samples(generator, latent_dim, n_samples):
	# generate points in latent space
	x_input = generate_latent_points(latent_dim, n_samples)
	# predict outputs
	X = generator.predict(x_input)
	# create class labels
	y = zeros((n_samples, 1))
	return X, y

# generate samples and save as a plot and save the model
def summarize_performance(step, g_model, latent_dim, n_samples=100):
	# prepare fake examples
	X, _ = generate_fake_samples(g_model, latent_dim, n_samples)
	# scale from [-1,1] to [0,1]
	X = (X + 1) / 2.0
	# plot images
	for i in range(10 * 10):
		# define subplot
		pyplot.subplot(10, 10, 1 + i)
		# turn off axis
		pyplot.axis('off')
		# plot raw pixel data
		pyplot.imshow(X[i, :, :, 0], cmap='gray_r')
	# save plot to file
	pyplot.savefig('results_collapse/generated_plot_%03d.png' % (step+1))
	pyplot.close()
	# save the generator model
	g_model.save('results_collapse/model_%03d.h5' % (step+1))

# create a line plot of loss for the gan and save to file
def plot_history(d1_hist, d2_hist, g_hist, a1_hist, a2_hist):
	# plot loss
	pyplot.subplot(2, 1, 1)
	pyplot.plot(d1_hist, label='d-real')
	pyplot.plot(d2_hist, label='d-fake')
	pyplot.plot(g_hist, label='gen')
	pyplot.legend()
	# plot discriminator accuracy
	pyplot.subplot(2, 1, 2)
	pyplot.plot(a1_hist, label='acc-real')
	pyplot.plot(a2_hist, label='acc-fake')
	pyplot.legend()
	# save plot to file
	pyplot.savefig('results_collapse/plot_line_plot_loss.png')
	pyplot.close()

# train the generator and discriminator
def train(g_model, d_model, gan_model, dataset, latent_dim, n_epochs=10, n_batch=128):
	# calculate the number of batches per epoch
	bat_per_epo = int(dataset.shape[0] / n_batch)
	# calculate the total iterations based on batch and epoch
	n_steps = bat_per_epo * n_epochs
	# calculate the number of samples in half a batch
	half_batch = int(n_batch / 2)
	# prepare lists for storing stats each iteration
	d1_hist, d2_hist, g_hist, a1_hist, a2_hist = list(), list(), list(), list(), list()
	# manually enumerate epochs
	for i in range(n_steps):
		# get randomly selected 'real' samples
		X_real, y_real = generate_real_samples(dataset, half_batch)
		# update discriminator model weights
		d_loss1, d_acc1 = d_model.train_on_batch(X_real, y_real)
		# generate 'fake' examples
		X_fake, y_fake = generate_fake_samples(g_model, latent_dim, half_batch)
		# update discriminator model weights
		d_loss2, d_acc2 = d_model.train_on_batch(X_fake, y_fake)
		# prepare points in latent space as input for the generator
		X_gan = generate_latent_points(latent_dim, n_batch)
		# create inverted labels for the fake samples
		y_gan = ones((n_batch, 1))
		# update the generator via the discriminator's error
		g_loss = gan_model.train_on_batch(X_gan, y_gan)
		# summarize loss on this batch
		print('>%d, d1=%.3f, d2=%.3f g=%.3f, a1=%d, a2=%d' %
			(i+1, d_loss1, d_loss2, g_loss, int(100*d_acc1), int(100*d_acc2)))
		# record history
		d1_hist.append(d_loss1)
		d2_hist.append(d_loss2)
		g_hist.append(g_loss)
		a1_hist.append(d_acc1)
		a2_hist.append(d_acc2)
		# evaluate the model performance every 'epoch'
		if (i+1) % bat_per_epo == 0:
			summarize_performance(i, g_model, latent_dim)
	plot_history(d1_hist, d2_hist, g_hist, a1_hist, a2_hist)

# make folder for results
makedirs('results_collapse', exist_ok=True)
# size of the latent space
latent_dim = 1
# create the discriminator
discriminator = define_discriminator()
# create the generator
generator = define_generator(latent_dim)
# create the gan
gan_model = define_gan(generator, discriminator)
# load image data
dataset = load_real_samples()
print(dataset.shape)
# train model
train(generator, discriminator, gan_model, dataset, latent_dim)

---

➡️ **Next / 下一步**: File 3 of 4